<center><b><font size='10'><font color='78aa58'>G</font><font color='6589bf'>V</font>HMR</font></b></center>

<center><b><font size='5'> World-Grounded Human Motion Recovery via <font color='78aa58'>Gravity</font>-<font color='6589bf'>View</font> <font color='c68b5b'>Coordinates</font></font></b></center>

![](https://zju3dv.github.io/gvhmr/img/teaser_v1.png)


<center><b>
<a href=https://zju3dv.github.io/gvhmr>Project Page</a>
|
<a href=https://arxiv.org/abs/2409.06662>Paper</a>
</b></center>

> World-Grounded Human Motion Recovery via Gravity-View Coordinates  
> [Zehong Shen](https://zehongs.github.io/)<sup>\*</sup>,
[Huaijin Pi](https://phj128.github.io/)<sup>\*</sup>,
[Yan Xia](https://isshikihugh.github.io/scholar),
[Zhi Cen](https://scholar.google.com/citations?user=Xyy-uFMAAAAJ),
[Sida Peng](https://pengsida.net/)<sup>†</sup>,
[Zechen Hu](https://zju3dv.github.io/gvhmr),
[Hujun Bao](http://www.cad.zju.edu.cn/home/bao/),
[Ruizhen Hu](https://csse.szu.edu.cn/staff/ruizhenhu/),
[Xiaowei Zhou](https://xzhou.me/)  
> SIGGRAPH Asia 2024


## Installation

Check [INSTALL.md](https://github.com/IsshikiHugh/GVHMR/blob/main/docs/INSTALL.md) if you want to install GVHMR on your own machine.

> Tips: you can fold the section and run the whole installation section at once.

In [3]:
# Controlling notebook is connected to NVIDIA drivers with CUDA. If this doesn't load check that GPU is selected as hardware accelerator under Edit -> Notebook settings.
# Google may shut down the GPU if usage has surpassed the allocation

!nvidia-smi

Wed Nov 18 04:03:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Environment Prepration ~ 15 min

Check [INSTALL.md#Environment](https://github.com/IsshikiHugh/GVHMR/blob/main/docs/INSTALL.md#environment) for further information.

In [4]:
import os
from pathlib import Path

# Clone the repo.
!git clone https://github.com/zju3dv/GVHMR
proj_root = str(Path('GVHMR').absolute())

# Install GVHMR
# FIX: Colab runs Python 3.13, but GVHMR needs Python 3.10.
# GVHMR is installed into a seperate Python 3.10 environment (/content/py310)
%cd {proj_root}
py310 = '/content/py310/bin/python'
!pip install -q uv
!uv venv --seed --python 3.10 /content/py310

# chumpy's setup.py imports pip
!grep -v chumpy requirements.txt > /tmp/requirements_py310.txt

#setuptools<81: newer setuptools removed pkg_resources, which lightning 2.3 needs
!uv pip install --python {py310} --index-strategy unsafe-best-match -r /tmp/requirements_py310.txt "setuptools<81"

#pip installed separetely (without build isolation)
!uv pip install --python {py310} --no-build-isolation chumpy
!uv pip install --python {py310} -e .

Cloning into 'GVHMR'...
remote: Enumerating objects: 378, done.
remote: Counting objects: 100% (209/209), done.
remote: Compressing objects: 100% (156/156), done.
remote: Total 378 (delta 71), reused 53 (delta 53), pack-reused 169 (from 1)
Receiving objects: 100% (378/378), 9.31 MiB | 14.89 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/GVHMR
Using CPython 3.10.22
Creating virtual environment with seed packages at: /content/py310
 + packaging==26.3
 + pip==26.3
 + setuptools==84.2.1
 + wheel==0.48.0
Activate with: source /content/py310/bin/activate
Using Python 3.10.22 environment at: /content/py310
Resolved 217 packages in 10.03s
Prepared 216 packages in 1m 11s
Uninstalled 2 packages in 14ms
Installed 217 packages in 1.12s
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.6
 + annotated-types==0.8.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.16.0
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==26.2.0
 + arrow==1.4.0
 + asttokens==3.

In [ ]:
# Install DPVO. (Optional)
#%cd third-party/DPVO

#!wget https://gitlab.com/libeigen/eigen/-/archive/3.4.0/eigen-3.4.0.zip
#!unzip -o eigen-3.4.0.zip -d thirdparty && rm -rf eigen-3.4.0.zip

#%pip install torch-scatter -f "https://data.pyg.org/whl/torch-2.3.0+cu121.html"
#%pip install numba pypose

#if 'cuda_home' not in locals():
#  cuda_home = '/usr/local/cuda-12'
#  if not Path(cuda_home).exists():
#    raise FileNotFoundError('CUDA_HOME for cuda 12.x not found!')

# os.environ['CUDA_HOME'] = cuda_home
#  os.environ['PATH'] = os.environ['PATH'] + f':{cuda_home}/bin'

# %pip install -e .
# %cd {proj_root}

### Data Prepration ~ 1 min

Check [INSTALL.md#Inputs&Outputs](https://github.com/IsshikiHugh/GVHMR/blob/main/docs/INSTALL.md#inputs--outputs) for further information.

In [35]:
%cd {proj_root}
!mkdir inputs
!mkdir outputs

!apt install -y -qq aria2
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPL_NEUTRAL.pkl -d inputs/checkpoints/body_models/smpl -o SMPL_NEUTRAL.pkl
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPLX_NEUTRAL.npz -d inputs/checkpoints/body_models/smplx -o SMPLX_NEUTRAL.npz
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/GVHMR/resolve/main/dpvo/dpvo.pth -d inputs/checkpoints/dpvo -o dpvo.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/GVHMR/resolve/main/gvhmr/gvhmr_siga24_release.ckpt -d inputs/checkpoints/gvhmr -o gvhmr_siga24_release.ckpt
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/GVHMR/resolve/main/hmr2/epoch%3D10-step%3D25000.ckpt -d inputs/checkpoints/hmr2 -o epoch=10-step=25000.ckpt
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/GVHMR/resolve/main/vitpose/vitpose-h-multi-coco.pth -d inputs/checkpoints/vitpose -o vitpose-h-multi-coco.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/GVHMR/resolve/main/yolo/yolov8x.pt -d inputs/checkpoints/yolo -o yolov8x.pt


/content/GVHMR
mkdir: cannot create directory ‘inputs’: File exists
mkdir: cannot create directory ‘outputs’: File exists
aria2 is already the newest version (1.37.0+debian-1build3).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
c8962c|OK  |       0B/s|inputs/checkpoints/body_models/smpl/SMPL_NEUTRAL.pkl

Status Legend:
(OK):download completed.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
4f5cbf|OK  |       0B/s|inputs/checkpoints/body_models/smplx/SMPLX_NEUTRAL.npz

Status Legend:
(OK):download completed.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
443f27|OK  |       0B/s|inputs/checkpoints/dpvo/dpvo.pth

Status Legend:
(OK):download completed.

Download Results:
gi

In [14]:
%cd /content/GVHMR

# Create the RICH directory
!mkdir -p inputs/RICH

# Download RICH support archive
!wget -c \
https://huggingface.co/camenduru/GVHMR/resolve/main/preproc_data/RICH_hmr4d_support.tar.gz \
-O RICH_hmr4d_support.tar.gz

# Extract into inputs/
!tar -xzvf RICH_hmr4d_support.tar.gz -C inputs/

# Verify the missing file
!ls -lh inputs/RICH/hmr4d_support/rich_test_labels.pt

/content/GVHMR
--2026-11-18 04:38:50--  https://huggingface.co/camenduru/GVHMR/resolve/main/preproc_data/RICH_hmr4d_support.tar.gz
Resolving huggingface.co (huggingface.co)... 65.8.76.91, 65.8.76.97, 65.8.76.109, ...
Connecting to huggingface.co (huggingface.co)|65.8.76.91|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/66e46d1b6e6ce3af7296f357/487d802beb27ee772ff831d23ebd8112d2a453371662c4318c7000c618223f43?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27RICH_hmr4d_support.tar.gz%3B+filename%3D%22RICH_hmr4d_support.tar.gz%22%3B&X-Xet-Cas-Uid=public&xip=fy8QB9Xq0e8&user_id=public&response-content-type=application%2Fgzip&Expires=1794980330&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjZlNDZkMWI2ZTZjZTNhZjcyOTZmMzU3LzQ4N2Q4MDJiZWIyN2VlNzcyZmY4MzFkMjNlYmQ4MTEyZDJhNDUzMzcxNjYyYzQzMThjNzAwMGM2MTgyMjNmNDNcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomWC1YZXQtQ2

## Run Demo

Use `-s` to skip visual odometry <b><u>if you know the camera is static</u></b>, otherwise the camera will be estimated by DPVO.

We also provide a script demo_folder.py to inference a entire folder.


```shell
python tools/demo/demo.py --video=docs/example_video/tennis.mp4 -s
python tools/demo/demo_folder.py -f inputs/demo/folder_in -d outputs/demo/folder_out -s

```

In [6]:
import io
import base64
from IPython.display import HTML
import cv2 # FIX: Hmr4d is installed in the Python 3.10 env

def display_video(fn):
    cap = cv2.VideoCapture(fn)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    scale = min(W, 1080) / W
    W, H = int(W * scale), int(H * scale)
    video_encoded = base64.b64encode(io.open(fn, 'rb').read())
    return HTML(data='''<video width="{0}" height="{1}" alt="test" controls>
                        <source src="data:video/mp4;base64,{2}" type="video/mp4" />
                        </video>'''.format(W, H, video_encoded.decode('ascii')))

In [11]:
!ls -lah inputs/RICH/
!ls -lah inputs/RICH/hmr4d_support/

ls: cannot access 'inputs/RICH/': No such file or directory
ls: cannot access 'inputs/RICH/hmr4d_support/': No such file or directory


### Demo Tennis

In [16]:
# Run demo.
video_fn = f'{proj_root}/docs/example_video/tennis.mp4'

# FIX: run demo in Python 3.10 environment
!PYTHONPATH= MPLBACKEND=Agg {py310} {proj_root}/tools/demo/demo.py --video={video_fn} -s

[11/18 04:40:50][INFO] [Input]: /content/GVHMR/docs/example_video/tennis.mp4
[11/18 04:40:50][INFO] (L, W, H) = (312, 812, 720)
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:212: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver(
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:218: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver(
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:230: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/ome

In [9]:
# Visualize the result.
display_video('outputs/demo/tennis/tennis_3_incam_global_horiz.mp4')

### Custom Demo

In order to run your own video, you need to follow instructions below:

0. Run the code block below to initialize `demo_google_drive_video()`.
1. Upload your video on Google drive and set accessbility as "Anyone with link (Viewer)".
2. Copy the link and extract the ID. The link should follow this pattern: `https://drive.google.com/file/d/<Google Drive ID>/view?usp=sharing`.
3. Call `demo_google_drive_video()` with the URL, and pass `True` if the camera is static.


In [17]:
%cd {proj_root}
!mkdir -p inputs/demo

def demo_google_drive_video(url:str, static_camera:bool):
    ''' URL should be like https://drive.google.com/file/d/xxxxxxxx/view?usp=drive_link '''

    print(f'[1/3] 📥 Downloading video...')
    gdid = url.split('/')[5]
    video_name = f'custom_{gdid}'
    download_url = f'\'https://drive.google.com/uc?id={gdid}&export=download&confirm=t\''
    !gdown {download_url} -O inputs/demo/{video_name}.mp4

    print(f'[2/3] 💃 Start running GVHMR...')
    flag = '-s' if static_camera else ''

    #point to py310
    !PYTHONPATH= MPLBACKEND=Agg {py310} {proj_root}/tools/demo/demo.py --video=inputs/demo/{video_name}.mp4 {flag}

    print(f'[3/3] 📺 Displaying result...')
    return display_video(f'outputs/demo/{video_name}/{video_name}_3_incam_global_horiz.mp4')

/content/GVHMR


In [18]:
demo_google_drive_video(url='https://drive.google.com/file/d/1KkTsoAuj9yq5JCJ-qg4_MnNBn56wR47F/view?usp=drive_link', static_camera=True)

[1/3] 📥 Downloading video...
Downloading...
From: https://drive.google.com/uc?id=1KkTsoAuj9yq5JCJ-qg4_MnNBn56wR47F&export=download&confirm=t
To: /content/GVHMR/inputs/demo/custom_1KkTsoAuj9yq5JCJ-qg4_MnNBn56wR47F.mp4
100% 936k/936k [00:00<00:00, 89.6MB/s]
[2/3] 💃 Start running GVHMR...
[11/18 04:44:06][INFO] [Input]: inputs/demo/custom_1KkTsoAuj9yq5JCJ-qg4_MnNBn56wR47F.mp4
[11/18 04:44:06][INFO] (L, W, H) = (367, 682, 666)
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:212: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver(
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:218: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 f

In [22]:
%cd {proj_root}
import re

def demo_uploaded_video(static_camera:bool, path:str=None):
  if path is None:
    from google.colab import files
    print(f'[1/3] 📤 Choose your video from your video...')
    src = list(files.upload().keys())[0]
  else:
    print(f'[1/3] 📤 Using {path}')
    src = path
  video_name = 'upload_' + re.sub(r'[^A-Za-z0-9]+', '_', Path(src).stem).strip('_')
  !mkdir -p inputs/demo && cp "{src}" inputs/demo/{video_name}.mp4 && rm -rf outputs/demo/{video_name}

  print(f'[2/3] 💃 Start running GVHMR...')
  flag = '-s' if static_camera else ''
  !PYTHONPATH= MPLBACKEND=Agg {py310} {proj_root}/tools/demo/demo.py --video=inputs/demo/{video_name}.mp4 {flag}

  print(f'[3/3] 📺 Displaying result...')
  return display_video(f'outputs/demo/{video_name}/{video_name}_3_incam_global_horiz.mp4')



/content/GVHMR


In [23]:
# Try it yourself!
demo_uploaded_video(static_camera=False)

[1/3] 📤 Choose your video from your video...


Saving runner2.mp4 to runner2.mp4
[2/3] 💃 Start running GVHMR...
[11/18 04:53:03][INFO] [Input]: inputs/demo/upload_runner2.mp4
[11/18 04:53:03][INFO] (L, W, H) = (0, 2160, 3840)
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:212: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver(
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:218: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver(
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:230: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() 

In [31]:
%cd /content/GVHMR

# Create EMDB directory
!mkdir -p inputs/EMDB

# Download EMDB support archive
!wget -c \
https://huggingface.co/camenduru/GVHMR/resolve/main/preproc_data/EMDB_hmr4d_support.tar.gz \
-O EMDB_hmr4d_support.tar.gz

# Extract archive
!tar -xzvf EMDB_hmr4d_support.tar.gz -C inputs/

!mkdir -p inputs/3DPW

!wget -c \
https://huggingface.co/camenduru/GVHMR/resolve/main/preproc_data/3DPW_hmr4d_support.tar.gz \
-O 3DPW_hmr4d_support.tar.gz

!tar -xzvf 3DPW_hmr4d_support.tar.gz -C inputs/

%cd /content/GVHMR

!find inputs/checkpoints/body_models -maxdepth 3 -type f

/content/GVHMR
--2026-11-18 05:19:12--  https://huggingface.co/camenduru/GVHMR/resolve/main/preproc_data/EMDB_hmr4d_support.tar.gz
Resolving huggingface.co (huggingface.co)... 13.35.202.40, 13.35.202.34, 13.35.202.97, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.40|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/66e46d1b6e6ce3af7296f357/465893d9a92f2f41a3735de46b094df641bc0807fce64f41873a2a7f18ae5e74?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27EMDB_hmr4d_support.tar.gz%3B+filename%3D%22EMDB_hmr4d_support.tar.gz%22%3B&response-content-type=application%2Fgzip&X-Xet-Cas-Uid=public&xip=fy8QB9Xq0e8&user_id=public&Expires=1794982752&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjZlNDZkMWI2ZTZjZTNhZjcyOTZmMzU3LzQ2NTg5M2Q5YTkyZjJmNDFhMzczNWRlNDZiMDk0ZGY2NDFiYzA4MDdmY2U2NGY0MTg3M2EyYTdmMThhZTVlNzRcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomcmV

In [40]:
%cd /content/GVHMR

# Create the required directory
!mkdir -p inputs/checkpoints/body_models/smplx

# Download SMPL-X body models
!wget -c https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPLX_MALE.npz \
  -O inputs/checkpoints/body_models/smplx/SMPLX_MALE.npz

!wget -c https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPLX_FEMALE.npz \
  -O inputs/checkpoints/body_models/smplx/SMPLX_FEMALE.npz

!wget -c https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPLX_NEUTRAL.npz \
  -O inputs/checkpoints/body_models/smplx/SMPLX_NEUTRAL.npz


/content/GVHMR
--2026-11-18 05:27:48--  https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPLX_MALE.npz
Resolving huggingface.co (huggingface.co)... 13.35.202.40, 13.35.202.97, 13.35.202.121, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.40|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/651b94abb9ae9f37afb56f82/537306cc40255fdf7cd753931355227e445d0a31af17b353c9bdb4d0c45fcfd5?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27SMPLX_MALE.npz%3B+filename%3D%22SMPLX_MALE.npz%22%3B&user_id=public&X-Xet-Cas-Uid=public&xip=fy8QB9Xq0e8&Expires=1794983268&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjUxYjk0YWJiOWFlOWYzN2FmYjU2ZjgyLzUzNzMwNmNjNDAyNTVmZGY3Y2Q3NTM5MzEzNTUyMjdlNDQ1ZDBhMzFhZjE3YjM1M2M5YmRiNGQwYzQ1ZmNmZDVcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomdXNlcl9pZD1wdWJsaWMmWC1YZXQtQ2FzLVVpZD1wdWJsaWMmeGlwPWZ5OFFCOVhxMGU4IiwiQ29uZGl0aW9uIj

In [41]:
%cd /content/GVHMR

# Create SMPL folder
!mkdir -p inputs/checkpoints/body_models/smpl

# Download male model
!wget -c https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPL_MALE.pkl \
  -O inputs/checkpoints/body_models/smpl/SMPL_MALE.pkl

# Download female model
!wget -c https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPL_FEMALE.pkl \
  -O inputs/checkpoints/body_models/smpl/SMPL_FEMALE.pkl

# Download neutral model
!wget -c https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPL_NEUTRAL.pkl \
  -O inputs/checkpoints/body_models/smpl/SMPL_NEUTRAL.pkl

/content/GVHMR
--2026-11-18 05:27:57--  https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPL_MALE.pkl
Resolving huggingface.co (huggingface.co)... 13.35.202.121, 13.35.202.97, 13.35.202.40, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.121|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/651b94abb9ae9f37afb56f82/81e1c9bda0b09e9091d4aa05b4b242ec1d44883a393b159cedaca595762cc772?X-Xet-Cas-Uid=public&xip=fy8QB9Xq0e8&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27SMPL_MALE.pkl%3B+filename%3D%22SMPL_MALE.pkl%22%3B&Expires=1794983277&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjUxYjk0YWJiOWFlOWYzN2FmYjU2ZjgyLzgxZTFjOWJkYTBiMDllOTA5MWQ0YWEwNWI0YjI0MmVjMWQ0NDg4M2EzOTNiMTU5Y2VkYWNhNTk1NzYyY2M3NzJcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ4aXA9Znk4UUI5WHEwZTgmdXNlcl9pZD1wdWJsaWMmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qIiwiQ29uZGl0aW9uIjp7

In [42]:
!PYTHONPATH= {py310} tools/train.py global/task=gvhmr/test_3dpw_emdb_rich exp=gvhmr/mixed/mixed ckpt_path=inputs/checkpoints/gvhmr/gvhmr_siga24_release.ckpt

/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:212: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver(
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:218: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver(
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:230: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver(
/content/py310/lib/python3.10/site-packages/hy

In [43]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r outputs/test_mocap/mixed \
    /content/drive/MyDrive/GVHMR_Benchmark_Results

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
